# Stroke Prediction Neural Network

## set up Packages

In [59]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

## set training variables

In [60]:
args = {}
args['batch_size'] = 64
args['test_batch_size'] = 64
args['epochs'] = 10
args['lr'] = 0.001
args['momentum'] = 0.5
args['seed'] = 1
args['device'] = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
args['log_interval'] = 10

## Load dataset

In [61]:
import pandas as pd

# Import Data
df = pd.read_csv('data/healthcare-dataset-stroke-data.csv')
df = df.drop(columns=['id'])
print(df.head())

   gender   age  hypertension  heart_disease ever_married      work_type  \
0    Male  67.0             0              1          Yes        Private   
1  Female  61.0             0              0          Yes  Self-employed   
2    Male  80.0             0              1          Yes        Private   
3  Female  49.0             0              0          Yes        Private   
4  Female  79.0             1              0          Yes  Self-employed   

  Residence_type  avg_glucose_level   bmi   smoking_status  stroke  
0          Urban             228.69  36.6  formerly smoked       1  
1          Rural             202.21   NaN     never smoked       1  
2          Rural             105.92  32.5     never smoked       1  
3          Urban             171.23  34.4           smokes       1  
4          Rural             174.12  24.0     never smoked       1  


## split data into training and test set

In [62]:
import numpy as np
from torch.utils.data import DataLoader, random_split, TensorDataset, Dataset

X_dataframe = df.iloc[:, :-1].copy() #features
Y_dataframe = df.iloc[:, -1].copy() #labels

#convert categorical columns into numerical for the model
X_dataframe = pd.get_dummies(X_dataframe, drop_first=True)

#force numeric values
X_dataframe = X_dataframe.apply(pd.to_numeric, errors='coerce').fillna(0.0)

Y_dataframe = pd.to_numeric(Y_dataframe, errors='coerce').fillna(0.0).astype(np.int64)

X_tensor = torch.from_numpy(X_dataframe.to_numpy(dtype=np.float32))
Y_tensor = torch.from_numpy(np.asarray(Y_dataframe, dtype=np.int64))
full_dataset = TensorDataset(X_tensor, Y_tensor)

print(f"Features Shape: {X_tensor.shape}", f"Labels Shape: {Y_tensor.shape}")

generator = torch.Generator().manual_seed(args['seed'])
train_dataset, test_dataset = random_split(full_dataset, [0.8, 0.2],generator=generator)

train_loader = DataLoader(train_dataset, batch_size=args['batch_size'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=args['test_batch_size'], shuffle=False)

print(f"Training Samples: {len(train_dataset)}, Test Samples: {len(test_dataset)}")

Features Shape: torch.Size([5110, 16]) Labels Shape: torch.Size([5110])
Training Samples: 4088, Test Samples: 1022


## Create NN

In [63]:

class StrokePredictionNet(nn.Module):
    def __init__(self):
        super(StrokePredictionNet, self).__init__()
        self.fc1 = nn.Linear(16, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)


    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x).squeeze(1)



## Train the Model

In [64]:

def train(epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        if args['device'] == 'cuda':
            data, target = data.cuda(), target.cuda()

        optimizer.zero_grad()
        output = model(data)

        loss = F.binary_cross_entropy_with_logits(output, target.float().view(-1))
        loss.backward()
        optimizer.step()

        if batch_idx % args['log_interval'] == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

## Evaluate the model

In [67]:
def test():
    model.eval()
    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            if args['device'] == 'cuda':
                data, target = data.cuda(), target.cuda()


            output = model(data)
            test_loss += F.binary_cross_entropy_with_logits(output, target.float().view(-1), reduction='sum').item()  # sum up batch loss
            pred = (torch.sigmoid(output) >= 0.5).long()
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

## Full training and test of model

In [68]:
model = StrokePredictionNet()
if args['device'] == 'cuda':
    model.cuda()

optimizer = optim.SGD(model.parameters(), lr=args['lr'], momentum=args['momentum'])

for epoch in range(1, args['epochs']+1):
    train(epoch)
    test()

Train Epoch: 1 [0/4088 (0%)]	Loss: 0.172489
Train Epoch: 1 [640/4088 (16%)]	Loss: 0.353795
Train Epoch: 1 [1280/4088 (31%)]	Loss: 0.290205
Train Epoch: 1 [1920/4088 (47%)]	Loss: 0.198626
Train Epoch: 1 [2560/4088 (62%)]	Loss: 0.289611
Train Epoch: 1 [3200/4088 (78%)]	Loss: 0.165993
Train Epoch: 1 [3840/4088 (94%)]	Loss: 0.178032

Test set: Average loss: 0.2056, Accuracy: 980/1022 (96%)

Train Epoch: 2 [0/4088 (0%)]	Loss: 0.258926
Train Epoch: 2 [640/4088 (16%)]	Loss: 0.302769
Train Epoch: 2 [1280/4088 (31%)]	Loss: 0.323733
Train Epoch: 2 [1920/4088 (47%)]	Loss: 0.284280
Train Epoch: 2 [2560/4088 (62%)]	Loss: 0.225058
Train Epoch: 2 [3200/4088 (78%)]	Loss: 0.337488
Train Epoch: 2 [3840/4088 (94%)]	Loss: 0.221429

Test set: Average loss: 0.1918, Accuracy: 980/1022 (96%)

Train Epoch: 3 [0/4088 (0%)]	Loss: 0.159378
Train Epoch: 3 [640/4088 (16%)]	Loss: 0.223859
Train Epoch: 3 [1280/4088 (31%)]	Loss: 0.352101
Train Epoch: 3 [1920/4088 (47%)]	Loss: 0.160519
Train Epoch: 3 [2560/4088 (62%)]	